In [ ]:
-- ============================================================
-- 1. LOCAL RANK: Rank subcells within their Survey Unit
--    and Survey Units within their Tile
-- ============================================================
CREATE OR REPLACE VIEW LOCAL_RANK_VIEW AS
SELECT
    EASTING,
    NORTHING,
    SENSOR_VALUE,
    CONVERT_XY(
        EASTING, NORTHING,
        2180160.001, 6660000.000,  -- origin x, y
        32.81,                      -- survey unit size in ft
        21, 18,                     -- tile grid: 21 wide x 18 tall
        10                          -- subcell grid: 10x10
    ) AS grid,
    grid['tile']::STRING          AS tile,
    grid['survey_unit']::INTEGER  AS survey_unit,
    grid['subcell']::INTEGER      AS subcell,

    -- Rank subcells within their Survey Unit (hottest = rank 1)
    RANK() OVER (
        PARTITION BY grid['tile']::STRING, grid['survey_unit']::INTEGER
        ORDER BY SENSOR_VALUE DESC
    ) AS subcell_rank_in_su,

    -- Rank survey units within their Tile (hottest = rank 1)
    RANK() OVER (
        PARTITION BY grid['tile']::STRING
        ORDER BY SENSOR_VALUE DESC
    ) AS su_rank_in_tile

FROM data5035.spring26.sdg_001_ra226_scandata;


-- ============================================================
-- 2. STABILITY SCORE: 1 / (1 + STDDEV) at the Survey Unit level
--    Higher score = more stable/consistent readings
-- ============================================================
CREATE OR REPLACE VIEW STABILITY_SCORE_VIEW AS
SELECT
    grid['tile']::STRING         AS tile,
    grid['survey_unit']::INTEGER AS survey_unit,
    COUNT(*)                     AS reading_count,
    AVG(SENSOR_VALUE)            AS avg_sensor_value,
    STDDEV(SENSOR_VALUE)         AS stddev_sensor_value,

    -- Stability Score: higher = more consistent readings
    1.0 / (1.0 + STDDEV(SENSOR_VALUE)) AS stability_score

FROM (
    SELECT
        SENSOR_VALUE,
        CONVERT_XY(
            EASTING, NORTHING,
            2180160.001, 6660000.000,
            32.81,
            21, 18,
            10
        ) AS grid
    FROM data5035.spring26.sdg_001_ra226_scandata
)
GROUP BY
    grid['tile']::STRING,
    grid['survey_unit']::INTEGER;